# Windows Copilot usability: aspect-sentiment clustering

Analysis of Windows Copilot for the usability evaluation in "When Convenience Meets Risk: Evaluating Usability,
Security, and AI Reliability Across Modern Operating Systems" (Alhawarat and Nijem).

**Data.** The AI-app review corpus of Chhetri et al. (2025, "What Users Value and Critique: Large-Scale Analysis of
User Feedback on AI-Powered Mobile Apps", IEEE Big Data), files `ai-summary-sentiment-merged-se3.parquet` and
`ai-aspect_extraction-pre-processed.parquet`, obtained from its authors on request. The corpus is not redistributed.

**Method.** Filters the corpus to the Microsoft Copilot apps, computes sentiment statistics, and clusters the extracted
aspect phrases into themes separately for positive and negative sentiment, following the original paper's procedure
(`preprocess.py`, `final_pos_2.py`, `final_neg_2.py`), followed by average-linkage merging of redundant clusters.

**Runtime.** About 5-10 minutes on a CPU runtime for roughly 20,000 phrases; a GPU speeds up the embedding step.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/My Drive/OS_Trends_NFR/UsbilityData"
OUT_DIR  = "/content/drive/My Drive/OS_Trends_NFR/UsbilityData/copilot_clustering_output"

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)


### Locate the data files

Prints the folder tree under the data directory and searches it for `ai-summary-sentiment-merged-se3.parquet`.
If the file is not found automatically, set `MERGED_FILE_PATH` in the next cell.

In [ ]:
import os

print(f"Contents of {DATA_DIR}:\n")
for root, dirs, files in os.walk(DATA_DIR):
    if 'copilot_clustering_output' in root:
        continue
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or '.'}/")
    for f in files:
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024*1024)
        print(f"{indent}  {f}  ({size_mb:.1f} MB)")


In [52]:
!pip install -q sentence-transformers pyarrow


In [ ]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter

import torch
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, silhouette_samples
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

pd.set_option("display.max_colwidth", 120)


## 2. Load data and identify Copilot rows

The merged file has one row per (review, extracted aspect) pair: a review mentioning two AI-feature aspects produces
two rows. `keys` is the short (at most three words) aspect summary and `values` its sentiment (`positive` or
`negative`), as produced by the original paper's extraction prompt.

In [ ]:
MERGED_FILE_PATH = None
TARGET_FILENAME = "ai-summary-sentiment-merged-se3.parquet"

for root, dirs, files in os.walk(DATA_DIR):
    if 'copilot_clustering_output' in root:
        continue
    if TARGET_FILENAME in files:
        MERGED_FILE_PATH = os.path.join(root, TARGET_FILENAME)
        break

if MERGED_FILE_PATH is None:
    raise FileNotFoundError(
        f"Could not auto-locate {TARGET_FILENAME} under {DATA_DIR}. "
        "Check the folder listing printed above and set MERGED_FILE_PATH manually, e.g.:\n"
        f'MERGED_FILE_PATH = "{DATA_DIR}/ai-related-dataset/{TARGET_FILENAME}"'
    )

print("Using merged file:", MERGED_FILE_PATH)

df = pd.read_parquet(MERGED_FILE_PATH)
print("Total rows in merged dataset:", len(df))
print("Total unique apps:", df['app_name'].nunique())
df.head(3)


In [ ]:
copilot_mask = df['app_name'].str.contains('Copilot', case=False, na=False)
copilot_apps = df.loc[copilot_mask, 'app_name'].value_counts()
print("Apps matching 'Copilot':")
print(copilot_apps)


In [ ]:
TARGET_APPS = ['\u200b\u200bMicrosoft Copilot', 'Microsoft Copilot', 'Microsoft 365 Copilot']

copilot_df = df[df['app_name'].isin(TARGET_APPS)].copy()
print("Rows for genuine Microsoft Copilot apps:", len(copilot_df))
print("Unique reviews:", copilot_df['rev_id'].nunique())
print()
print("Breakdown by app_name:")
print(copilot_df['app_name'].value_counts())


## 3. Overall sentiment statistics

In [ ]:
sentiment_counts = copilot_df['values'].value_counts()
sentiment_pct = copilot_df['values'].value_counts(normalize=True) * 100

print("Aspect-sentiment pair counts:")
print(sentiment_counts)
print()
print("Aspect-sentiment pair percentages:")
print(sentiment_pct.round(1))
print()
print(f"TOTAL aspect-sentiment pairs: {len(copilot_df)}")
print(f"TOTAL unique reviews contributing: {copilot_df['rev_id'].nunique()}")


In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sentiment_pct.reindex(['positive','negative']).plot(kind='bar', color=['#4472C4','#C00000'], ax=ax)
ax.set_ylabel('% of aspect-sentiment pairs')
ax.set_title('Windows Copilot: AI-Feature Sentiment Distribution')
ax.set_xticklabels(['Positive','Negative'], rotation=0)
for i, v in enumerate(sentiment_pct.reindex(['positive','negative'])):
    ax.text(i, v+1, f"{v:.1f}%", ha='center')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/copilot_sentiment_distribution.png", dpi=200)
plt.show()


## 4. Prepare text for clustering

Splits the aspect phrases (`keys`) by sentiment, following `preprocess.py`: lowercasing, removal of stray brackets
and quotes left by JSON parsing, and removal of empty or duplicate phrases.

In [ ]:
def clean_phrase(s):
    if not isinstance(s, str):
        return ""
    s = s.strip().strip('"').strip("'")
    s = re.sub(r'\s+', ' ', s)
    return s.lower()

copilot_df['keys_clean'] = copilot_df['keys'].apply(clean_phrase)
copilot_df = copilot_df[copilot_df['keys_clean'].str.len() > 0]

pos_phrases = copilot_df.loc[copilot_df['values']=='positive', 'keys_clean'].tolist()
neg_phrases = copilot_df.loc[copilot_df['values']=='negative', 'keys_clean'].tolist()

print(f"Positive phrases: {len(pos_phrases)}")
print(f"Negative phrases: {len(neg_phrases)}")
print()
print("Sample positive:", pos_phrases[:8])
print("Sample negative:", neg_phrases[:8])


## 5. Sentence embeddings

`sentence-transformers/all-mpnet-base-v2`, the embedding model of the original `preprocess.py`. Embeddings are
cached so that the step is not repeated on later runs.

In [ ]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=device)

def get_embeddings(phrases, cache_name):
    cache_path = f"{OUT_DIR}/{cache_name}.npy"
    if os.path.exists(cache_path):
        print(f"Loading cached embeddings from {cache_path}")
        return np.load(cache_path)
    print(f"Encoding {len(phrases)} phrases for {cache_name}...")
    emb = model.encode(phrases, show_progress_bar=True, batch_size=64)
    np.save(cache_path, emb)
    return emb

pos_embeddings = get_embeddings(pos_phrases, "copilot_pos_embeddings")
neg_embeddings = get_embeddings(neg_phrases, "copilot_neg_embeddings")

print("Positive embeddings shape:", pos_embeddings.shape)
print("Negative embeddings shape:", neg_embeddings.shape)


## 6. K-means sweep with evaluation metrics

Following `final_pos_2.py` and `final_neg_2.py`: for each k, reports the silhouette score, Davies-Bouldin index,
Calinski-Harabasz index, inertia, and average within-cluster cosine similarity, as the basis for choosing k.

In [ ]:
def cluster_similarity(embeddings, labels, k):
    sims = {}
    all_sims = []
    for i in range(k):
        idx = np.where(labels == i)[0]
        if len(idx) <= 1:
            sims[i] = 0.0
            continue
        sim_matrix = cosine_similarity(embeddings[idx])
        mask = np.triu(np.ones(sim_matrix.shape), k=1).astype(bool)
        pair_sims = sim_matrix[mask]
        all_sims.extend(pair_sims)
        sims[i] = float(np.mean(pair_sims))
    overall = float(np.mean(all_sims)) if all_sims else 0.0
    return sims, overall

def run_kmeans_sweep(embeddings, k_range=range(2, 31), random_state=42):
    rows = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        labels = km.fit_predict(embeddings)
        sil = silhouette_score(embeddings, labels)
        db = davies_bouldin_score(embeddings, labels)
        ch = calinski_harabasz_score(embeddings, labels)
        _, avg_sim = cluster_similarity(embeddings, labels, k)
        rows.append({
            'k': k, 'silhouette': sil, 'davies_bouldin': db,
            'calinski_harabasz': ch, 'inertia': km.inertia_,
            'avg_cosine_similarity': avg_sim
        })
        print(f"k={k:2d}  silhouette={sil:.4f}  davies_bouldin={db:.4f}  "
              f"calinski_harabasz={ch:.1f}  avg_sim={avg_sim:.4f}")
    return pd.DataFrame(rows)

print("=== POSITIVE phrases sweep ===")
pos_sweep = run_kmeans_sweep(pos_embeddings)


In [ ]:
print("=== NEGATIVE phrases sweep ===")
neg_sweep = run_kmeans_sweep(neg_embeddings)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(pos_sweep['k'], pos_sweep['silhouette'], marker='o', color='#4472C4')
axes[0].set_title('Positive phrases: silhouette vs k')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Silhouette score')
axes[1].plot(neg_sweep['k'], neg_sweep['silhouette'], marker='o', color='#C00000')
axes[1].set_title('Negative phrases: silhouette vs k')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette score')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/silhouette_vs_k.png", dpi=200)
plt.show()

pos_sweep.to_csv(f"{OUT_DIR}/pos_kmeans_sweep_metrics.csv", index=False)
neg_sweep.to_csv(f"{OUT_DIR}/neg_kmeans_sweep_metrics.csv", index=False)
print("Best k by silhouette -- positive:", int(pos_sweep.loc[pos_sweep['silhouette'].idxmax(),'k']))
print("Best k by silhouette -- negative:", int(neg_sweep.loc[neg_sweep['silhouette'].idxmax(),'k']))


## 7. Final clustering at the chosen k

`POS_K = 18` and `NEG_K = 20`. For negative phrases, k = 20 is a genuine interior maximum of the silhouette score
in the sweep. For positive phrases there is no clear interior peak; k = 18 reaches 94% of the maximum silhouette
score observed (at k = 30) with far fewer clusters, and matches the number of positive themes Chhetri et al. (2025)
report for their full corpus.

In [ ]:
POS_K = 18
NEG_K = 20
print(f"Using POS_K={POS_K}, NEG_K={NEG_K}")
print("(Fixed choice -- see comment above for the statistical + Chhetri-reference justification.)")

def final_cluster(embeddings, k, random_state=42):
    km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    labels = km.fit_predict(embeddings)
    return labels

pos_labels = final_cluster(pos_embeddings, POS_K)
neg_labels = final_cluster(neg_embeddings, NEG_K)


## 8. Theme labels

For each cluster, the most frequent distinctive unigrams and bigrams (stopwords excluded) and representative
phrases, used to assign short theme names in the manner of the original paper.

In [ ]:
def summarize_clusters(phrases, labels, k, n_samples=15, n_top_terms=8):
    summary = []
    for c in range(k):
        idx = np.where(labels == c)[0]
        cluster_phrases = [phrases[i] for i in idx]
        size = len(cluster_phrases)

        try:
            vec = CountVectorizer(ngram_range=(1,2), stop_words='english', max_features=50)
            X = vec.fit_transform(cluster_phrases)
            term_counts = np.asarray(X.sum(axis=0)).flatten()
            terms = vec.get_feature_names_out()
            top_terms = [t for t,_ in sorted(zip(terms, term_counts), key=lambda x: -x[1])[:n_top_terms]]
        except ValueError:
            top_terms = []

        samples = list(np.random.RandomState(42).choice(cluster_phrases, min(n_samples, size), replace=False))

        summary.append({
            'cluster': c,
            'size': size,
            'pct_of_total': round(100*size/len(phrases), 2),
            'top_terms': ', '.join(top_terms),
            'sample_phrases': samples
        })
    return pd.DataFrame(summary).sort_values('size', ascending=False).reset_index(drop=True)

pos_summary = summarize_clusters(pos_phrases, pos_labels, POS_K)
neg_summary = summarize_clusters(neg_phrases, neg_labels, NEG_K)

print("="*100)
print("POSITIVE CLUSTERS")
print("="*100)
for _, row in pos_summary.iterrows():
    print(f"\nCluster {row['cluster']}  |  size={row['size']} ({row['pct_of_total']}%)")
    print(f"  Top terms: {row['top_terms']}")
    print(f"  Samples: {row['sample_phrases'][:8]}")


In [ ]:
print("="*100)
print("NEGATIVE CLUSTERS")
print("="*100)
for _, row in neg_summary.iterrows():
    print(f"\nCluster {row['cluster']}  |  size={row['size']} ({row['pct_of_total']}%)")
    print(f"  Top terms: {row['top_terms']}")
    print(f"  Samples: {row['sample_phrases'][:8]}")


## 8.1 Merging redundant clusters (centroid similarity)

Some raw clusters express the same idea (for example, several variants of general praise), consistent with the low
silhouette scores of Section 6. Each cluster's centroid (mean embedding) and the pairwise cosine similarity between
centroids are computed, and the similarity matrix is plotted.

In [ ]:
def compute_centroids(embeddings, labels, k):
    return np.array([embeddings[labels == c].mean(axis=0) for c in range(k)])

pos_centroids = compute_centroids(pos_embeddings, pos_labels, POS_K)
neg_centroids = compute_centroids(neg_embeddings, neg_labels, NEG_K)

pos_centroid_sim = cosine_similarity(pos_centroids)
neg_centroid_sim = cosine_similarity(neg_centroids)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(pos_centroid_sim, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[0].set_title(f'Positive cluster centroid similarity (k={POS_K})')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Cluster')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(neg_centroid_sim, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[1].set_title(f'Negative cluster centroid similarity (k={NEG_K})')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Cluster')
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/centroid_similarity_heatmaps.png", dpi=200)
plt.show()

def offdiag_values(sim_matrix):
    n = sim_matrix.shape[0]
    mask = ~np.eye(n, dtype=bool)
    return sim_matrix[mask]

pos_offdiag = offdiag_values(pos_centroid_sim)
neg_offdiag = offdiag_values(neg_centroid_sim)

print("Positive centroid pairwise similarity: "
      f"min={pos_offdiag.min():.3f}  median={np.median(pos_offdiag):.3f}  "
      f"90th pct={np.percentile(pos_offdiag,90):.3f}  max={pos_offdiag.max():.3f}")
print("Negative centroid pairwise similarity: "
      f"min={neg_offdiag.min():.3f}  median={np.median(neg_offdiag):.3f}  "
      f"90th pct={np.percentile(neg_offdiag,90):.3f}  max={neg_offdiag.max():.3f}")


### Merge rule: average-linkage hierarchical clustering

Clusters are merged by average-linkage hierarchical clustering on centroid cosine distance: a merge is accepted only
if the average similarity across the resulting group clears the threshold. Transitive (single-linkage, union-find)
merging was rejected because the similarity matrix has no clean gaps, which makes it prone to chaining.
`DISTANCE_THRESHOLD` is in units of 1 - similarity (0.45 corresponds to a similarity of 0.55); the dendrogram
after the merge cell shows where it cuts the tree.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform

def agglomerative_merge(sim_matrix, distance_threshold, method='average'):
    n = sim_matrix.shape[0]
    dist_matrix = 1 - sim_matrix
    np.fill_diagonal(dist_matrix, 0)
    dist_matrix = (dist_matrix + dist_matrix.T) / 2
    condensed = squareform(dist_matrix, checks=False)
    Z = linkage(condensed, method=method)
    labels = fcluster(Z, t=distance_threshold, criterion='distance')
    groups = {}
    for i, lab in enumerate(labels):
        groups.setdefault(lab, []).append(i)
    return list(groups.values()), Z

DISTANCE_THRESHOLD = 0.45
LINKAGE_METHOD = 'average'

pos_merge_groups, pos_Z = agglomerative_merge(pos_centroid_sim, DISTANCE_THRESHOLD, LINKAGE_METHOD)
neg_merge_groups, neg_Z = agglomerative_merge(neg_centroid_sim, DISTANCE_THRESHOLD, LINKAGE_METHOD)

print(f"Positive: {POS_K} original clusters -> {len(pos_merge_groups)} merged meta-themes")
for g in pos_merge_groups:
    if len(g) > 1:
        print(f"  MERGED: clusters {g}")

print(f"\\nNegative: {NEG_K} original clusters -> {len(neg_merge_groups)} merged meta-themes")
for g in neg_merge_groups:
    if len(g) > 1:
        print(f"  MERGED: clusters {g}")

for name, groups, k in [("Positive", pos_merge_groups, POS_K), ("Negative", neg_merge_groups, NEG_K)]:
    biggest = max(len(g) for g in groups)
    if biggest > k / 2:
        print(f"\\nWARNING: {name} has a meta-theme containing {biggest}/{k} original clusters -- "
              f"still likely too aggressive. Try lowering DISTANCE_THRESHOLD and re-running.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
dendrogram(pos_Z, ax=axes[0], color_threshold=DISTANCE_THRESHOLD)
axes[0].axhline(y=DISTANCE_THRESHOLD, color='red', linestyle='--', label=f'cut = {DISTANCE_THRESHOLD}')
axes[0].set_title(f'Positive clusters dendrogram ({LINKAGE_METHOD}-linkage)')
axes[0].set_xlabel('Original cluster index'); axes[0].set_ylabel('Distance (1 - similarity)')
axes[0].legend()

dendrogram(neg_Z, ax=axes[1], color_threshold=DISTANCE_THRESHOLD)
axes[1].axhline(y=DISTANCE_THRESHOLD, color='red', linestyle='--', label=f'cut = {DISTANCE_THRESHOLD}')
axes[1].set_title(f'Negative clusters dendrogram ({LINKAGE_METHOD}-linkage)')
axes[1].set_xlabel('Original cluster index'); axes[1].set_ylabel('Distance (1 - similarity)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/merge_dendrograms.png", dpi=200)
plt.show()

print("If the red line crosses too many branches at once (over-merging) or too few (under-")
print("merging) relative to what looks reasonable in the tree structure, adjust DISTANCE_THRESHOLD")
print("in the previous cell and re-run both cells.")


### Summaries for the merged themes

Top terms and representative phrases, as in Section 8, computed for the merged groups.

In [ ]:
def summarize_merged(phrases, labels, merge_groups, n_samples=15, n_top_terms=8):
    summary = []
    for meta_id, group in enumerate(merge_groups):
        idx = np.concatenate([np.where(labels == c)[0] for c in group])
        cluster_phrases = [phrases[i] for i in idx]
        size = len(cluster_phrases)
        try:
            vec = CountVectorizer(ngram_range=(1,2), stop_words='english', max_features=50)
            X = vec.fit_transform(cluster_phrases)
            term_counts = np.asarray(X.sum(axis=0)).flatten()
            terms = vec.get_feature_names_out()
            top_terms = [t for t,_ in sorted(zip(terms, term_counts), key=lambda x: -x[1])[:n_top_terms]]
        except ValueError:
            top_terms = []
        samples = list(np.random.RandomState(42).choice(cluster_phrases, min(n_samples, size), replace=False))
        summary.append({
            'meta_cluster': meta_id, 'original_clusters': group, 'size': size,
            'pct_of_total': round(100*size/len(phrases), 2),
            'top_terms': ', '.join(top_terms), 'sample_phrases': samples
        })
    return pd.DataFrame(summary).sort_values('size', ascending=False).reset_index(drop=True)

pos_merged_summary = summarize_merged(pos_phrases, pos_labels, pos_merge_groups)
neg_merged_summary = summarize_merged(neg_phrases, neg_labels, neg_merge_groups)

print("="*100)
print(f"POSITIVE MERGED META-THEMES ({len(pos_merge_groups)} total)")
print("="*100)
for _, row in pos_merged_summary.iterrows():
    tag = f" [merged from clusters {row['original_clusters']}]" if len(row['original_clusters']) > 1 else ""
    print(f"\nMeta-cluster {row['meta_cluster']}{tag}  |  size={row['size']} ({row['pct_of_total']}%)")
    print(f"  Top terms: {row['top_terms']}")
    print(f"  Samples: {row['sample_phrases'][:8]}")


In [ ]:
print("="*100)
print(f"NEGATIVE MERGED META-THEMES ({len(neg_merge_groups)} total)")
print("="*100)
for _, row in neg_merged_summary.iterrows():
    tag = f" [merged from clusters {row['original_clusters']}]" if len(row['original_clusters']) > 1 else ""
    print(f"\nMeta-cluster {row['meta_cluster']}{tag}  |  size={row['size']} ({row['pct_of_total']}%)")
    print(f"  Top terms: {row['top_terms']}")
    print(f"  Samples: {row['sample_phrases'][:8]}")


## 9. Theme names

Human-readable theme names (2-4 words) assigned by the authors to each merged theme after reviewing its terms and
phrases. A theme left as `"TODO"` falls back to its top terms in the export.

In [ ]:
pos_theme_names = {i: "TODO" for i in range(len(pos_merge_groups))}
neg_theme_names = {i: "TODO" for i in range(len(neg_merge_groups))}

print("Positive theme names:", pos_theme_names)
print("Negative theme names:", neg_theme_names)


## 10. Export

In [ ]:
def finalize(summary_df, theme_names, sentiment_label):
    out = summary_df.copy()
    out['theme_name'] = out['meta_cluster'].map(theme_names)
    out.loc[out['theme_name']=='TODO', 'theme_name'] = out['top_terms']
    out['sentiment'] = sentiment_label
    out['sample_phrases'] = out['sample_phrases'].apply(lambda lst: '; '.join(lst[:5]))
    out['merged_from_clusters'] = out['original_clusters'].apply(lambda g: ','.join(map(str, g)))
    return out[['sentiment','meta_cluster','theme_name','size','pct_of_total',
                 'merged_from_clusters','top_terms','sample_phrases']]

pos_final = finalize(pos_merged_summary, pos_theme_names, 'positive')
neg_final = finalize(neg_merged_summary, neg_theme_names, 'negative')

final_table = pd.concat([pos_final, neg_final], ignore_index=True)
final_table.to_csv(f"{OUT_DIR}/copilot_theme_clusters_final.csv", index=False)

print(f"Saved final MERGED theme table to {OUT_DIR}/copilot_theme_clusters_final.csv")
print(f"({len(pos_final)} positive meta-themes, {len(neg_final)} negative meta-themes -- "
      f"down from {POS_K} and {NEG_K} raw clusters)")
final_table


In [ ]:
summary_stats = {
    'app_names_included': TARGET_APPS,
    'total_aspect_sentiment_pairs': int(len(copilot_df)),
    'unique_reviews': int(copilot_df['rev_id'].nunique()),
    'positive_pairs': int(sentiment_counts.get('positive', 0)),
    'negative_pairs': int(sentiment_counts.get('negative', 0)),
    'positive_pct': round(float(sentiment_pct.get('positive', 0)), 2),
    'negative_pct': round(float(sentiment_pct.get('negative', 0)), 2),
    'positive_k_clusters_raw': POS_K,
    'negative_k_clusters_raw': NEG_K,
    'positive_meta_themes_after_merge': len(pos_merge_groups),
    'negative_meta_themes_after_merge': len(neg_merge_groups),
    'merge_distance_threshold': DISTANCE_THRESHOLD,
    'merge_linkage_method': LINKAGE_METHOD,
}
with open(f"{OUT_DIR}/copilot_summary_stats.json", "w") as f:
    json.dump(summary_stats, f, indent=2)

print(json.dumps(summary_stats, indent=2))
print(f"\nAll outputs saved to: {OUT_DIR}")


## Notes

- `copilot_summary_stats.json`: overall positive/negative split, sample sizes, and the number of raw clusters and
  merged themes.
- `copilot_theme_clusters_final.csv`: one row per merged theme, with size, share, the raw clusters merged into it,
  and representative phrases.
- The merged themes, not the raw k-means clusters, are the unit reported in the paper (11 positive, 15 negative).
- Embeddings and sweep metrics are cached, so re-running the notebook reuses them.